# DocVLM-Fingerprint: Open-Source VLM Kaggle Case Study

This notebook runs a paper-facing open VLM comparison through the same DocVLM-Fingerprint pipeline:

- `Qwen/Qwen3-VL-8B-Instruct`
- `OpenGVLab/InternVL3_5-8B-HF`
- `Qwen/Qwen3-VL-4B-Instruct`

The first two models are distinct 8B-class open VLM families. The 4B Qwen model adds a balanced scale-family comparison against Qwen3-VL-8B without making the project a broad benchmark. `zai-org/GLM-4.1V-9B-Thinking` remains configured for optional appendix experiments only.

The run evaluates clean and perturbed document-centric examples, splits model answers into claims, scores support against evidence, regenerates metrics/plots, and exports a results zip.


In [ ]:
MODEL_NAMES = ["qwen3_vl_8b", "internvl35_8b_hf", "qwen3_vl_4b"]


RUN_SMOKE_TESTS = True
SMOKE_LIMIT = 1
RUN_FULL_EVAL = True
FULL_EVAL_LIMIT = None  
CLEAR_OPENAI_KEY_FOR_LOCAL_SCORING = True


In [ ]:

%pip uninstall -y torchaudio
%pip install -q -U --no-deps git+https://github.com/huggingface/transformers
%pip install -q -U "safetensors>=0.8.0" accelerate qwen-vl-utils sentencepiece timm einops


In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

WORK_DIR = Path("/kaggle/working")
INPUT_DIR = Path("/kaggle/input")
PROJECT_DIR = WORK_DIR / "docvlm-fingerprint"

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

zip_candidates = sorted(INPUT_DIR.rglob("docvlm-fingerprint-kaggle-dataset.zip"))
if not zip_candidates:
    zip_candidates = sorted(INPUT_DIR.rglob("*.zip"))

if zip_candidates:
    project_zip = zip_candidates[0]
    print(f"Using dataset zip: {project_zip}")
    with zipfile.ZipFile(project_zip, "r") as archive:
        archive.extractall(WORK_DIR)
else:
    extracted_candidates = [
        path for path in sorted(INPUT_DIR.rglob("docvlm-fingerprint"))
        if (path / "src" / "evaluate.py").exists()
    ]
    if not extracted_candidates:
        raise FileNotFoundError(
            "Could not find the project zip or extracted docvlm-fingerprint folder. "
            "Attach the Kaggle dataset named docvlm-fingerprint-kaggle-dataset to this notebook."
        )
    source_project = extracted_candidates[0]
    print(f"Using extracted Kaggle dataset folder: {source_project}")
    shutil.copytree(source_project, PROJECT_DIR)

if not (PROJECT_DIR / "src" / "evaluate.py").exists():
    raise FileNotFoundError(f"Project is missing src/evaluate.py under {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")

# Start from a clean runtime-results directory so exports cannot contain stale checked-in outputs.
results_dir = PROJECT_DIR / "results"
if results_dir.exists():
    for child in results_dir.iterdir():
        if child.name == "external":
            # External checks are managed by separate cells/scripts; keep them out of the main paper export path.
            shutil.rmtree(child, ignore_errors=True)
        elif child.is_dir():
            shutil.rmtree(child, ignore_errors=True)
        else:
            child.unlink()
results_dir.mkdir(parents=True, exist_ok=True)
print("Cleared runtime results directory:", results_dir)



In [ ]:
# Defensive source hotfixes for stale Kaggle dataset uploads.
# If you uploaded an older dataset zip but copied newer notebook cells, patch the working copy here.
from pathlib import Path

metrics_path = PROJECT_DIR / "src" / "metrics.py"
metrics_text = metrics_path.read_text(encoding="utf-8")
if "def read_jsonl(path: Path, *, allow_empty: bool = False)" not in metrics_text:
    old_empty_block = '''    if not records:
        raise ValueError(f"required input is empty: {path.relative_to(ROOT_DIR)}")'''
    new_empty_block = '''    if not records and not allow_empty:
        raise ValueError(f"required input is empty: {path.relative_to(ROOT_DIR)}")'''
    metrics_text = metrics_text.replace(
        "def read_jsonl(path: Path) -> list[dict[str, Any]]:",
        "def read_jsonl(path: Path, *, allow_empty: bool = False) -> list[dict[str, Any]]:",
    )
    metrics_text = metrics_text.replace(old_empty_block, new_empty_block)
    metrics_text = metrics_text.replace(
        "scored_outputs = read_jsonl(SCORED_OUTPUTS_PATH)",
        "scored_outputs = read_jsonl(SCORED_OUTPUTS_PATH, allow_empty=True)",
    )
    metrics_path.write_text(metrics_text, encoding="utf-8")
    print("Patched stale src/metrics.py to allow empty scored_outputs during smoke tests.")
else:
    print("src/metrics.py already has empty-claim smoke-test handling.")

vlm_clients_path = PROJECT_DIR / "src" / "vlm_clients.py"
vlm_clients_text = vlm_clients_path.read_text(encoding="utf-8")
if "import re\n" not in vlm_clients_text.split("from pathlib import Path", 1)[0]:
    vlm_clients_text = vlm_clients_text.replace("import os\nimport time\n", "import os\nimport re\nimport time\n")
    vlm_clients_path.write_text(vlm_clients_text, encoding="utf-8")
    print("Patched stale src/vlm_clients.py to import re for answer cleanup.")

old_image_source = '''                    {
                        "type": "image",
                        self.config.get("image_content_key", "url"): _resolved_image_uri(image_path),
                    },'''
new_image_source = '''                    {
                        "type": "image",
                        self.config.get("image_content_key", "image"): _resolved_image_path(image_path),
                    },'''
if old_image_source in vlm_clients_text:
    vlm_clients_text = vlm_clients_text.replace(old_image_source, new_image_source)
    vlm_clients_path.write_text(vlm_clients_text, encoding="utf-8")
    print("Patched stale src/vlm_clients.py to pass local image paths instead of file:// URIs.")
elif "_resolved_image_uri(image_path)" in vlm_clients_text:
    print("Warning: src/vlm_clients.py still contains _resolved_image_uri(image_path). Update the dataset zip or patch manually.")
else:
    print("src/vlm_clients.py already passes local image paths to HF processors.")

models_path = PROJECT_DIR / "configs" / "models.yaml"
models_text = models_path.read_text(encoding="utf-8")
missing = []
for model_name in ["qwen3_vl_8b", "internvl35_8b_hf", "qwen3_vl_4b", "glm41v_9b_thinking"]:
    marker = f"  - name: {model_name}\n"
    if marker not in models_text:
        continue
    start = models_text.index(marker)
    end = models_text.find("\n  - name:", start + 1)
    if end == -1:
        end = len(models_text)
    block = models_text[start:end]
    if "    image_content_key: image\n" not in block:
        missing.append(model_name)
if missing:
    print("Warning: these model configs are missing image_content_key: image:", missing)
else:
    print("configs/models.yaml has image_content_key: image for generic HF VLMs.")

# Hard protocol verification for the paper-facing rerun.
models_config_path = PROJECT_DIR / "configs" / "models.yaml"
evaluate_path = PROJECT_DIR / "src" / "evaluate.py"
clients_path = PROJECT_DIR / "src" / "vlm_clients.py"
models_config_text = models_config_path.read_text(encoding="utf-8")
evaluate_text = evaluate_path.read_text(encoding="utf-8")
clients_text = clients_path.read_text(encoding="utf-8")
required_protocol_checks = {
    "configs/models.yaml has two-line prompt": "Return exactly two lines" in models_config_text,
    "src/evaluate.py parses Answer line": "def extract_final_answer" in evaluate_text,
    "src/evaluate.py stores claim_source": "claim_source" in evaluate_text,
    "src/vlm_clients.py preserves protocol lines": "has_protocol" in clients_text,
}
missing_protocol = [name for name, ok in required_protocol_checks.items() if not ok]
if missing_protocol:
    raise RuntimeError(
        "The mounted Kaggle project is stale and does not contain the Answer/Evidence protocol: "
        + "; ".join(missing_protocol)
        + ". Upload the latest kaggle/docvlm-fingerprint-kaggle-dataset.zip and restart the session."
    )
print("Answer/Evidence protocol verified in mounted source.")



In [ ]:
# Prevent accidental paid/API-backed scoring paths. The open-source VLM clients do not need OpenAI keys.
if CLEAR_OPENAI_KEY_FOR_LOCAL_SCORING:
    os.environ.pop("OPENAI_API_KEY", None)

!python src/dataset.py
!python -m unittest discover -s tests
!python src/evidence_scorer.py --self-test

In [ ]:
import subprocess
import json
import gc
from pathlib import Path
import shutil

RESULTS_DIR = PROJECT_DIR / "results"
PER_MODEL_DIR = RESULTS_DIR / "per_model"
PER_MODEL_DIR.mkdir(parents=True, exist_ok=True)

STANDARD_OUTPUTS = [
    "raw_outputs.jsonl",
    "claim_outputs.jsonl",
    "scored_outputs.jsonl",
    "metrics.csv",
    "failure_examples.jsonl",
]


def run_command(args: list[str]) -> None:
    print("$ " + " ".join(args))
    subprocess.run(args, cwd=PROJECT_DIR, check=True)

def verify_protocol_outputs(model_name: str) -> None:
    raw_path = RESULTS_DIR / "raw_outputs.jsonl"
    claim_path = RESULTS_DIR / "claim_outputs.jsonl"
    raw_rows = [json.loads(line) for line in raw_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    claim_rows = [json.loads(line) for line in claim_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    answer_line_count = sum("Answer:" in str(row.get("answer", "")) for row in raw_rows)
    evidence_line_count = sum("Evidence:" in str(row.get("answer", "")) for row in raw_rows)
    claim_source_count = sum(bool(str(row.get("claim_source", "")).strip()) for row in claim_rows)
    parsed_answer_count = sum("parsed_answer" in row for row in raw_rows)
    print(
        f"protocol check for {model_name}: "
        f"Answer lines={answer_line_count}/{len(raw_rows)}, "
        f"Evidence lines={evidence_line_count}/{len(raw_rows)}, "
        f"parsed_answer fields={parsed_answer_count}/{len(raw_rows)}, "
        f"claim_source fields={claim_source_count}/{len(claim_rows)}"
    )
    if parsed_answer_count != len(raw_rows) or claim_source_count != len(claim_rows):
        raise RuntimeError("This run used stale evaluation code; upload the latest dataset zip and restart Kaggle.")
    if evidence_line_count == 0:
        sample = raw_rows[0].get("answer", "") if raw_rows else ""
        raise RuntimeError(
            "The model outputs did not include Evidence: lines." + repr(sample[:300])
        )


def clear_standard_outputs() -> None:
    for name in STANDARD_OUTPUTS:
        path = RESULTS_DIR / name
        if path.exists():
            path.unlink()


def clear_runtime_caches() -> None:
    cache_dir = RESULTS_DIR / "cache"
    for name in ["model_response_cache.jsonl", "claim_splitter_cache.jsonl", "claim_scoring_cache.jsonl"]:
        path = cache_dir / name
        if path.exists():
            path.unlink()


def save_model_outputs(model_name: str, tag: str) -> None:
    for name in ["raw_outputs.jsonl", "claim_outputs.jsonl", "scored_outputs.jsonl"]:
        source = RESULTS_DIR / name
        if not source.exists():
            raise FileNotFoundError(f"Expected output missing after {model_name}: {source}")
        shutil.copy2(source, PER_MODEL_DIR / f"{source.stem}_{tag}.jsonl")


def run_model(model_name: str, limit: int | None, tag: str) -> None:
    clear_standard_outputs()
    clear_runtime_caches()
    args = ["python", "src/evaluate.py", "--models", model_name]
    if limit is not None:
        args.extend(["--limit", str(limit)])
    run_command(args)
    verify_protocol_outputs(model_name)
    run_command(["python", "src/metrics.py"])
    save_model_outputs(model_name, tag)
    gc.collect()


def merge_jsonl(input_paths: list[Path], output_path: Path) -> None:
    with output_path.open("w", encoding="utf-8") as output:
        for path in input_paths:
            with path.open("r", encoding="utf-8") as handle:
                for line in handle:
                    if line.strip():
                        output.write(line)

print("Ready to run models:", MODEL_NAMES)
print("Full evaluation limit:", FULL_EVAL_LIMIT)

In [ ]:
# Smoke test each model on a tiny subset first. If a model OOMs, use one of the fallback MODEL_NAMES lists above.
if RUN_SMOKE_TESTS:
    for model_name in MODEL_NAMES:
        print(f"\n=== Smoke test: {model_name} ===")
        run_model(model_name=model_name, limit=SMOKE_LIMIT, tag=f"smoke_{model_name}")


In [ ]:
# Full per-model runs. Each run overwrites standard output files, then copies them to results/per_model/.
if RUN_FULL_EVAL:
    completed = []
    for model_name in MODEL_NAMES:
        print(f"\n=== Full evaluation: {model_name} ===")
        run_model(model_name=model_name, limit=FULL_EVAL_LIMIT, tag=model_name)
        completed.append(model_name)
else:
    completed = MODEL_NAMES

print("Completed models:", completed)


In [ ]:
# Merge per-model outputs back into the standard result files, then regenerate combined metrics/plots/report text.
source_tags = MODEL_NAMES if RUN_FULL_EVAL else [f"smoke_{model}" for model in MODEL_NAMES]
merge_jsonl([PER_MODEL_DIR / f"raw_outputs_{tag}.jsonl" for tag in source_tags], RESULTS_DIR / "raw_outputs.jsonl")
merge_jsonl([PER_MODEL_DIR / f"claim_outputs_{tag}.jsonl" for tag in source_tags], RESULTS_DIR / "claim_outputs.jsonl")
merge_jsonl([PER_MODEL_DIR / f"scored_outputs_{tag}.jsonl" for tag in source_tags], RESULTS_DIR / "scored_outputs.jsonl")
run_command(["python", "src/metrics.py"])

!python - <<'PY'
import csv, json
from pathlib import Path
raw = [json.loads(line) for line in Path('results/raw_outputs.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
scored = [json.loads(line) for line in Path('results/scored_outputs.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
metrics = list(csv.DictReader(open('results/metrics.csv', encoding='utf-8')))
print('raw outputs:', len(raw))
print('scored claims:', len(scored))
print('metric rows:', len(metrics))
print('models:', sorted({row['model'] for row in metrics}))
PY

In [ ]:
# Export all results for download from Kaggle.
archive_base = WORK_DIR / "docvlm_fingerprint_open_vlm_results"
archive_path = shutil.make_archive(str(archive_base), "zip", RESULTS_DIR)
print(f"Download this file from Kaggle output: {archive_path}")

## After the run

Download `docvlm_fingerprint_open_vlm_results.zip` from Kaggle outputs and copy the generated JSONL/CSV/PNG files back into your local `results/` folder.

Before updating the paper claims, manually inspect 20-30 rows from `results/per_model/scored_outputs_*.jsonl`, especially cases where answer accuracy and claim faithfulness disagree. The strongest paper angle is not simply which model wins; it is how Qwen3-VL, InternVL3.5, and GLM-4.1V fail differently under blur, crop/removal, JPEG compression, and distractor text.
